In [1]:
import os
import math
import json
import numpy as np
import random

# 런팟
os.environ["WANDB_PROJECT"] = "patent_disc"
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import load_dataset, load_from_disk
from sklearn.metrics import f1_score

In [2]:
# Config
SEARCH = False   # True=레시피 탐색(짧은 런) / False=최종 풀런

config = {
    "num_labels": 188,
    "seed": 42,
    "learning_rate": 4.8e-4,
    "epochs": 2 if SEARCH else 12,
    "early_stop": 6,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "model_name": "skt/A.X-Encoder-base",
    "max_len": 512,          
    "eff_batch": 128,           
    "micro_batch": 128,
    "eval_micro_batch": 512,
    "repo_final": "ingyoun/A.X-patent-len512-bce",
    "out_path": "/workspace/output/modernbert-len512_bce",
    "tag": "modernbert-patent-len512-bce",
    "rev": "9708f9c404ace91efd25c06fac2d73413616f4ef",
}

config["grad_accum"] = config["eff_batch"] // config["micro_batch"]          # micro×accum = eff_batch
config["run_name"] = (
    f"axenc_len512_bce"
    + ("_search" if SEARCH else "_full")
)

os.environ["WANDB_NOTEBOOK_NAME"] = os.path.abspath("09_03_Loss_BCE.ipynb")

In [3]:
random.seed(config['seed'])
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
torch.cuda.manual_seed_all(config['seed'])

In [4]:
print(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

cuda


## 데이터셋

In [5]:
dataset = load_dataset(
    "ingyoun/patent-clean-text-modernbert-tokenized",
    cache_dir="/workspace/hf_cache",
)

dataset

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11162
    })
})

## 모델

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model_name_or_path=config["model_name"],
        num_labels=config["num_labels"], 
        problem_type="multi_label_classification", 
        classifier_dropout=0.5,
        dtype=torch.float32,
        attn_implementation="flash_attention_2",            # len8192와 구현 일치
    )

[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertForSequenceClassification is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`
[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: skt/A.X-Encoder-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 토크나이저

In [7]:
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], revision=config["rev"])
EOS_ID = tokenizer.eos_token_id


def _prep(batch):
    max_len = config["max_len"]
    ids, masks = [], []
    for x, m in zip(batch["input_ids"], batch["attention_mask"]):
        if len(x) > max_len:
            x = x[: max_len - 1] + [EOS_ID]   # <s> 유지 + 꼬리를 <\s>로 마감
            m = m[:max_len]
        ids.append(x)
        masks.append(m)
    return {"input_ids": ids, "attention_mask": masks, "length": [len(i) for i in ids]}


# 절단은 max_len에만 의존 → 길이별로 볼륨에 캐시. 재훈련 시 절단(.map) 생략, max_len 변경 시 새 경로로 재생성
prep_cache = f"/workspace/prep_cache/len{config['max_len']}"
if os.path.isdir(prep_cache):
    dataset = load_from_disk(prep_cache)
    print(f"prep 캐시 로드: {prep_cache}")
else:
    dataset = dataset.map(_prep, batched=True)
    dataset.save_to_disk(prep_cache)
    print(f"prep 캐시 저장: {prep_cache}")

print(f"EOS_ID={EOS_ID}  max_len={config['max_len']}")
dataset

prep 캐시 로드: /workspace/prep_cache/len512
EOS_ID=1  max_len=512


DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11162
    })
})

## 커스텀

In [8]:
class BCELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, logits, labels) -> torch.Tensor:
        return F.binary_cross_entropy_with_logits(logits, labels, reduction="mean")

In [9]:
class BCETrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = BCELoss()

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = self.loss_fn(outputs.logits, labels.float())
        return (loss, outputs) if return_outputs else loss

In [10]:
class MultiLabelCollator:
    def __init__(self, tokenizer):
        self.tok = tokenizer
    
    def __call__(self, feats):
        labels = torch.tensor([f["labels"] for f in feats], dtype=torch.float)
        keys = ("input_ids", "attention_mask")
        enc = [{k: f[k] for k in keys if k in f} for f in feats]
        batch = self.tok.pad(enc, padding=True, return_tensors="pt")
        batch["labels"] = labels
        return batch

In [11]:
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def compute_metrics(eval_pred):
    """
    sigmoid→τ=0.5 멀티라벨 F1(micro/macro/sample).
    micro_f1이 모델 선택(metric_for_best_model) 기준. 벤더 연속성 앵커(top-1 weighted)도 병기.
    """
    logits, labels = eval_pred
    logits = np.asarray(logits)
    Y = np.asarray(labels).astype(int)
    pred = (_sigmoid(logits) >= 0.5).astype(int)
    return {
        "micro_f1":  f1_score(Y, pred, average="micro",   zero_division=0),   # headline (모델 선택 기준)
        "macro_f1":  f1_score(Y, pred, average="macro",   zero_division=0),
        "sample_f1": f1_score(Y, pred, average="samples", zero_division=0),
        "empty_rate": float((pred.sum(1) == 0).mean()),
        "anchor_weighted_f1": f1_score(Y.argmax(1), logits.argmax(1), average="weighted", zero_division=0),  # 벤더 연속성
    }

## 훈련

In [12]:
steps_per_epoch = math.ceil(len(dataset["train"]) / config["eff_batch"])
num = 4 if SEARCH else 2
eval_steps = math.ceil(steps_per_epoch / num)           # 탐색:에폭당 4회 / 풀런:에폭당 2회

training_args = TrainingArguments(
    output_dir='/app/results',
    seed=config["seed"],
    learning_rate=config["learning_rate"],
    weight_decay=config["weight_decay"],
    lr_scheduler_type="linear",
    warmup_ratio=config["warmup_ratio"],
    per_device_train_batch_size=config["micro_batch"],
    per_device_eval_batch_size=config["eval_micro_batch"],
    gradient_accumulation_steps=config["grad_accum"],   
    train_sampling_strategy="group_by_length",          # 유사 길이 배치로 padding 최소화
    remove_unused_columns=False,                        # 커스텀 collator가 키를 직접 선택 + length 컬럼 보존
    num_train_epochs=config["epochs"],
    bf16=True,                                          # ModernBERT 계열 안정성엔 bf16 (bf16=True, fp16=False)
    eval_strategy='steps',
    eval_steps=eval_steps,
    save_strategy="no" if SEARCH else "steps",          # 탐색:저장 안 함 / 풀런:에폭당 저장
    save_steps=eval_steps,                              # save_strategy="no"면 무시됨
    save_total_limit=3,
    logging_steps=50,
    metric_for_best_model="micro_f1",         
    greater_is_better=True,
    load_best_model_at_end=not SEARCH,                  # 풀런에서만 best 복원
    report_to="wandb",
    run_name=config["run_name"]
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [13]:
trainer = BCETrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    data_collator=MultiLabelCollator(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stop"])]
)

In [14]:
import gc


gc.collect()
torch.cuda.empty_cache()
print(f"잔여 allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB (모델 가중치)")

잔여 allocated: 0.6 GB (모델 가중치)


In [15]:
trainer.train()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
789,0.015946,0.013297,0.602735,0.550021,0.524289,0.345010,0.653598
1578,0.010698,0.010347,0.701005,0.669502,0.658574,0.193693,0.714413
2367,0.009129,0.008680,0.746491,0.720183,0.714779,0.149525,0.745044
3156,0.008842,0.008213,0.761252,0.742674,0.742708,0.104103,0.738117
3945,0.007752,0.007675,0.780678,0.769975,0.769271,0.085110,0.766859
4734,0.007307,0.007176,0.799594,0.789478,0.799175,0.057337,0.771873
5523,0.006632,0.007101,0.803838,0.793731,0.806777,0.047393,0.775966
6312,0.006832,0.007029,0.798008,0.785564,0.793677,0.070417,0.781293
7101,0.005813,0.006702,0.816314,0.807241,0.814279,0.057069,0.784930
7890,0.005930,0.006576,0.814093,0.805601,0.810980,0.059040,0.783625


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18936, training_loss=0.008017698724261935, metrics={'train_runtime': 30301.5072, 'train_samples_per_second': 79.954, 'train_steps_per_second': 0.625, 'total_flos': 7.510082862466372e+17, 'train_loss': 0.008017698724261935, 'epoch': 12.0})

## 평가

In [16]:
if not SEARCH:
    test_metrics = trainer.evaluate(dataset["test"], metric_key_prefix="test")
    for k, v in test_metrics.items():
        print(f"{k}: {v}")

[transformers] early stopping required metric_for_best_model, but did not find eval_micro_f1 so early stopping is disabled


Training Loss,Validation Loss,Step,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
0.000469,0.007721,18936,0.853795,0.850798,0.868932,0.010558,0.813301


test_loss: 0.007720820605754852
test_micro_f1: 0.85379513633014
test_macro_f1: 0.8507976438491988
test_sample_f1: 0.8689324900397565
test_empty_rate: 0.010558069381598794
test_anchor_weighted_f1: 0.8133008236363002


In [17]:
if not SEARCH:
    os.makedirs(config["out_path"], exist_ok=True)

    metrics_fp = os.path.join(config["out_path"], f"{config['tag']}_test_metrics.json")
    with open(metrics_fp, "w", encoding="utf-8") as f:
        json.dump(test_metrics, f, ensure_ascii=False, indent=2)

    print("saved", config["out_path"])

saved /workspace/output/modernbert-len512_bce


In [18]:
if not SEARCH:
    print(trainer.state.best_model_checkpoint)
    print(trainer.state.best_metric)

/app/results/checkpoint-18936
0.8580352494980293


In [ ]:
if not SEARCH:
    trainer.model.push_to_hub(config["repo_final"])
    tokenizer.push_to_hub(config["repo_final"])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/598M [00:00<?, ?B/s]